> Notebook-friendly copy of `part-I/1.8-statistical-foundations-and-ml-solutions.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# Solutions

**ℹ️ Reference solutions**

Worked solutions for the short exercises in [1.8-statistical-foundations-and-ml-exercises.ipynb](1.8-statistical-foundations-and-ml-exercises.ipynb).

## Exercise 1: Explore

Build a DataFrame of 60 stations with `elevation_m` uniform on [200, 3500] and `temp_celsius = 15 - 6.5 * elevation_m/1000 + noise` (noise standard deviation 1.5 °C, seeded). Make a seaborn regression plot of temperature against elevation and print `describe()`.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
elev = rng.uniform(200, 3500, 60)
temp = 15 - 6.5 * (elev / 1000) + rng.normal(0, 1.5, 60)
df = pd.DataFrame({"elevation_m": elev, "temp_celsius": temp})
sns.regplot(data=df, x="elevation_m", y="temp_celsius")
plt.show()
print(df.describe().round(2))

## Exercise 2: Correlation

Print the Pearson correlation matrix of the DataFrame and identify the sign of the temperature–elevation correlation.

In [ ]:
print(df.corr().round(3))   # temperature vs elevation is strongly negative (the lapse rate)

## Exercise 3: Fit a linear regression

Fit `LinearRegression` with `X = df[["elevation_m"]]` and `y = df["temp_celsius"]`. Print the recovered lapse rate in °C km⁻¹ and the intercept.

In [ ]:
from sklearn.linear_model import LinearRegression
X = df[["elevation_m"]]; y = df["temp_celsius"]
m = LinearRegression().fit(X, y)
print(round(m.coef_[0] * 1000, 2), "°C km^-1")
print(round(m.intercept_, 2), "°C")

## Exercise 4: Honest evaluation

Split the data (test size 0.3, fixed random state), fit on the training set, and report the test RMSE and R^2.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
m = LinearRegression().fit(Xtr, ytr)
p = m.predict(Xte)
print("RMSE:", round(root_mean_squared_error(yte, p), 3))
print("R^2: ", round(r2_score(yte, p), 3))

## Exercise 5: Multivariate regression on real advertising data

The data cached at `path` below (fetched by the pre-supplied cell) holds real advertising spend and sales for 200 markets: `TV`, `Radio`, and `Newspaper` budgets (thousands of dollars) and `Sales` (thousands of units) — the classic dataset behind the "which channel actually drives sales" question (James et al., *An Introduction to Statistical Learning*).

1. Load the CSV. Build `X` from all three budget columns and `y` from `Sales`.
2. Split into train and test sets (test size 0.3, fixed random state), fit a `LinearRegression`, and print the three coefficients alongside their column names.
3. Report the test RMSE and R^2. Which budget has the largest coefficient? Before concluding that channel is the most effective, what would you need to check about the three columns' scales?

In [ ]:
# Pre-supplied: download the data file and cache it locally.
# You do not need to understand this cell yet — fetching data is covered in the
# reproducible-data-pipelines bonus subchapter.
import pooch

path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/Advertising.csv",
    known_hash="sha256:69104adc017e75d7019f61fe66ca2eb4ab014ee6f2a9b39b452943f209352010",
    fname="advertising.csv",
    path=pooch.os_cache("mlees"),
)

In [ ]:
ads = pd.read_csv(path, index_col=0)     # the first column is a row number, not data
print(ads.head())

In [ ]:
X_ads = ads[["TV", "Radio", "Newspaper"]]
y_ads = ads["Sales"]

In [ ]:
Xtr_ads, Xte_ads, ytr_ads, yte_ads = train_test_split(
    X_ads, y_ads, test_size=0.3, random_state=0
)
ads_model = LinearRegression().fit(Xtr_ads, ytr_ads)

In [ ]:
for name, coef in zip(X_ads.columns, ads_model.coef_):
    print(f"{name:>10}: {coef:+.4f}")
print(f"{'intercept':>10}: {ads_model.intercept_:+.4f}")

In [ ]:
pred_ads = ads_model.predict(Xte_ads)
print("test RMSE:", round(root_mean_squared_error(yte_ads, pred_ads), 3))
print("test R^2: ", round(r2_score(yte_ads, pred_ads), 3))

In [ ]:
# Radio has the largest coefficient (+0.200 against TV's +0.044), but that does not make it
# the most effective channel: a coefficient is "sales per unit of budget", and the three
# budgets do not span the same range.
print(X_ads.describe().loc[["min", "max"]].round(1))

# TV runs to 296 while Radio stops at 49.6, so a one-unit change is a much smaller share of
# the Radio budget. Comparing effect sizes means comparing coefficients on standardised
# columns -- which is what the pipeline in the next exercise sets up.

## Exercise 6: A pipeline with scaling

Build a `Pipeline` of `StandardScaler` followed by `LinearRegression`, fit it on the training set, and print its test R^2 (via `.score`).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
print(round(pipe.score(Xte, yte), 3))

## Exercise 7: Demonstrate overfitting

Fit a degree-8 polynomial pipeline and compare its R^2 on training data with its R^2 on test data. `elevation_m` is in metres (order 10^2-10^3); an 8th-degree polynomial built directly on raw metres is numerically unstable rather than overfit, so rescale it to kilometres first (divide by 1000) — the same rescaling the lecture's own degree-8 demonstration relies on. A degree-8 polynomial has 9 parameters: fit on the full 42-point training set from Exercise 4, it is heavily over-determined and generalises fine instead of overfitting. Fit it on only the first 12 rows of the (already-shuffled) training set instead, matching the near-saturated regime — 12 points for 9 parameters — the lecture's own demonstration uses. Comment on the gap.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# elevation_m is in metres (order 1e3); an 8th-degree polynomial built directly on raw
# metres is numerically ill-conditioned rather than overfit, so rescale to kilometres
# first — the same fix the lecture's own degree-8 demonstration relies on
Xtr_km, Xte_km = Xtr / 1000, Xte / 1000

In [ ]:
# shrinking the training set alone (down to 9-12 points) only produces a mild gap here,
# because the elevations are scattered randomly across the full range rather than evenly
# spaced -- a sparse interpolation gap does not reliably force wild swings. Train on only
# the LOWEST 30% of the elevation range instead: the model never sees anything above
# ~1.2 km, so every test point above that requires extrapolation, and a degree-8
# polynomial's error grows explosively outside the range it was fit on
low_idx = Xtr_km[Xtr_km["elevation_m"] < Xtr_km["elevation_m"].quantile(0.3)].index
Xtr_km_small, ytr_small = Xtr_km.loc[low_idx], ytr.loc[low_idx]

In [ ]:
over = make_pipeline(PolynomialFeatures(degree=8), LinearRegression()).fit(Xtr_km_small, ytr_small)
print("train R^2:", round(over.score(Xtr_km_small, ytr_small), 3))
print("test  R^2:", round(over.score(Xte_km, yte), 3))
# a large train-test gap is overfitting: the flexible model memorised the low end of the
# range and never learned the shape of the data it was never shown

## Exercise 8: Cross-validation

Use `cross_val_score` with 5 folds to estimate the R^2 of a plain `LinearRegression` on the full dataset, and print the mean and standard deviation across folds.

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")
print(round(scores.mean(), 3), round(scores.std(), 3))

## Exercise 9: Cluster and reduce the penguins data

Using the same `palmer_penguins.csv` data as the lecture (fetched the same way), drop rows with missing measurements, then:

1. Cluster the penguins into 2 groups with `KMeans`, using `bill_length_mm` and `bill_depth_mm`. Print a crosstab of true species against cluster to see how well 2 clusters separate 3 species.
2. Reduce all four measurements (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`) to 2 components with `PCA`, after scaling them. Print the explained variance ratio.

In [ ]:
# Pre-supplied: download the data file and cache it locally.
# You do not need to understand this cell yet — fetching data is covered in the
# reproducible-data-pipelines bonus subchapter.
import pooch

penguins_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/palmer_penguins.csv",
    known_hash="sha256:f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93",
    fname="palmer_penguins.csv",
    path=pooch.os_cache("mlees"),
)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [ ]:
penguins = pd.read_csv(penguins_path).dropna(
    subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
).reset_index(drop=True)
print(penguins.shape)          # (342, 8): two rows carry no measurements at all

In [ ]:
# 1. two clusters from the two bill measurements
bill = penguins[["bill_length_mm", "bill_depth_mm"]]
km_bill = KMeans(n_clusters=2, n_init=10, random_state=0).fit(bill)
penguins["cluster"] = km_bill.labels_
print(pd.crosstab(penguins["species"], penguins["cluster"]))

# Two clusters cannot hold three species. Adelie lands almost entirely in cluster 0, while
# Chinstrap and Gentoo share cluster 1 -- k-means split the data where the gap between
# points is widest, and that is not where the third species boundary lies.

In [ ]:
# 2. four measurements, scaled, reduced to two components
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
scaled = StandardScaler().fit_transform(penguins[features])
pca = PCA(n_components=2).fit(scaled)

In [ ]:
print("explained variance ratio:", pca.explained_variance_ratio_.round(3))   # [0.688 0.193]
print("cumulative:", round(pca.explained_variance_ratio_.sum(), 3))          # 0.882

# Two components carry 88 % of the variance in four correlated measurements. Scaling first
# is what makes that meaningful: body_mass_g runs in thousands and bill_depth_mm in tens, so
# without it the first component would mostly describe the choice of units.

## Exercise 10: Forest fires, real data with seaborn

The data cached at `path` below (fetched by the pre-supplied cell) holds 517 real fire records from Montesinho natural park, Portugal (Cortez & Morais, 2007): each row is a fire with its month, weather conditions, and burned area.

1. Load the CSV. With `sns.boxplot`, compare `temp` (°C) across `month`.
2. With `sns.barplot`, compare mean `wind` (km/h) across `month`.
3. `area` (hectares burned) is heavily right-skewed, with many zero-area records. Plot its histogram once on all records, and once after filtering out the zero-area rows with a boolean mask. Which plot actually shows the shape of the fires that did burn?

In [ ]:
# Pre-supplied: download the data file and cache it locally.
# You do not need to understand this cell yet — fetching data is covered in the
# reproducible-data-pipelines bonus subchapter.
import pooch

path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/forest_fires.csv",
    known_hash="sha256:0d6586a1fa52f55bef48578aef14eb97273f1e9330e1a53423df497a77065253",
    fname="forest_fires.csv",
    path=pooch.os_cache("mlees"),
)

In [ ]:
fires = pd.read_csv(path)
print(fires.shape)             # (517, 13)

month_order = ["jan", "feb", "mar", "apr", "may", "jun",
               "jul", "aug", "sep", "oct", "nov", "dec"]

# 1. temperature by month
plt.figure(figsize=(9, 4))
sns.boxplot(data=fires, x="month", y="temp", order=month_order)
plt.xlabel("month")
plt.ylabel("temperature (degC)")
plt.title("Temperature at the time of each fire")
plt.show()

# 2. mean wind by month
plt.figure(figsize=(9, 4))
sns.barplot(data=fires, x="month", y="wind", order=month_order)
plt.xlabel("month")
plt.ylabel("wind speed (km/h)")
plt.title("Mean wind speed by month")
plt.show()

# 3. burned area, with and without the zero-area records
n_zero = (fires["area"] == 0.0).sum()
print(f"{n_zero} of {len(fires)} records burned no measurable area")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(fires["area"], bins=40, ax=axes[0])
axes[0].set_title(f"all {len(fires)} records")

burned = fires[fires["area"] > 0.0]     # boolean mask, as in 1.5
sns.histplot(burned["area"], bins=40, ax=axes[1])
axes[1].set_title(f"the {len(burned)} that burned")

for ax in axes:
    ax.set_xlabel("area burned (ha)")
plt.tight_layout()
plt.show()

# The second plot is the informative one. Nearly half the records are exact zeros, and in the
# first plot that spike is so tall it flattens everything else into the axis -- the shape of
# the distribution of fires that actually burned is invisible. The zeros are not missing data
# and should not be discarded from the dataset; they just belong to a different question
# ("did it burn?") from the one this histogram asks ("how much?").

## Exercise 11: Clustering the penguin data, at length

Exercise 9 clustered the penguins once, on the two bill measurements, and read the result off a
crosstab. This is the long version of the same question, on a different pair of measurements —
bill length against flipper length, beak against wing — and it asks the question Exercise 9 skipped:
how many clusters should there have been?

Nothing here needs a new dataset. The fetch cell from Exercise 9 already put `penguins_path` in
scope; if you are running this notebook from the top, reuse it.

**ℹ️ Beyond this subchapter**

One tool below is not demonstrated in 1.8's lecture:

- [`silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html) — Q3, a second opinion on the number of clusters, scoring how much closer each point sits to its own cluster than to the next nearest one

The elbow method, `KMeans`, `.inertia_`, `.fit`, `.predict` and `.labels_` are all from the lecture.

**Q1) Clean the data. Drop every row with a missing measurement.**

Call the result `penguin_df` and print its shape.

In [ ]:
measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguin_df = pd.read_csv(penguins_path).dropna(subset=measurements).reset_index(drop=True)
print(penguin_df.shape)        # (342, 8)

**Q2) Create an input array `X` from the `bill_length_mm` and `flipper_length_mm` columns.**

Its shape should be `(342, 2)`. Two columns, because the whole exercise is about being able to see
the answer on a flat plot.

In [ ]:
X = penguin_df[["bill_length_mm", "flipper_length_mm"]]
print(X.shape)                 # (342, 2)

**Q3) Train k-means over a range of k, and choose k twice — once by the elbow method and once
by silhouette analysis.**

For k from 1 to 9, fit `KMeans` and record `.inertia_`, the sum of squared distances from each point
to its assigned centre. Plot inertia against k.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.8-target-penguin-elbow.png" alt="Inertia falling steeply from k equals 1 to 2 and then flattening, forming an elbow" width="450">

<em>The elbow. Inertia always falls as k rises — with one cluster per point it reaches zero — so the
number you want is not the minimum but the bend, past which each extra cluster buys much less.</em>


Then, for k from 2 to 9, fit again and record `silhouette_score(X, labels)`. Plot that against k
too. Unlike inertia, the silhouette score has a genuine maximum.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.8-target-penguin-silhouette.png" alt="Silhouette score peaking sharply at k equals 2 and declining unevenly afterwards" width="450">

<em>The silhouette score, which peaks rather than flattens.</em>


In a comment, say which k each method points at, and whether they agree.

In [ ]:
from sklearn.metrics import silhouette_score

# the elbow: inertia against k
k_values = range(1, 10)
inertias = []
for k in k_values:
    inertias.append(KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).inertia_)

plt.figure(figsize=(5.5, 3.5))
plt.plot(list(k_values), inertias, marker="s", c="k", lw=2)
plt.xlabel("Number of clusters")
plt.ylabel("Sum of squared distances / inertia")
plt.title("Elbow method for optimal k")
plt.show()

print([round(v) for v in inertias])
# [77591, 21352, 14083, 9846, 7588, 6462, 5378, 4417, 3938]

# silhouette: only defined for k >= 2, since one cluster has nothing to compare against
silhouette_k = range(2, 10)
silhouettes = []
for k in silhouette_k:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).labels_
    silhouettes.append(silhouette_score(X, labels))

plt.figure(figsize=(5.5, 3.5))
plt.plot(list(silhouette_k), silhouettes, marker="s", c="k", lw=2)
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.title("Silhouette analysis")
plt.show()

print([round(v, 3) for v in silhouettes])
# [0.614, 0.484, 0.445, 0.426, 0.413, 0.431, 0.406, 0.394]

# The elbow is at k = 2: inertia falls from 77591 to 21352 and then only gently. The
# silhouette score peaks at k = 2 as well, at 0.614, and drops sharply afterwards. The two
# methods agree, and both point at two clusters -- for a dataset with three species.

**Q4) Cluster with k = 3 and compare the result against the true species.**

Fit `KMeans(n_clusters=3)`, predict the labels, and store them in `penguin_df` as a `cluster`
column. Then draw one scatter of bill length against flipper length in which the *marker shape*
carries the true species and the *colour* carries the predicted cluster — three
`plt.scatter` calls, one per species, each coloured by `c=subset["cluster"]`.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.8-target-penguin-kmeans-3.png" alt="Bill length against flipper length, markers by species and colours by predicted cluster, with Gentoo cleanly separated and Adelie and Chinstrap partly mixed" width="500">

<em>The figure to replicate. Where a marker shape and its colour agree everywhere, k-means found that
species; where one shape carries two colours, it did not.</em>


Hints: pass `cmap="cividis"` and fix the colour range with `vmin=0, vmax=2` on all three calls, or
the three subsets will each be scaled separately and the colours will not mean the same thing.
Use `s=` to vary the marker size and `label=` for the legend.

In a comment: which species does k-means recover, and which two does it confuse?

In [ ]:
kmeans_3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
penguin_df["cluster"] = kmeans_3.predict(X)

# marker shape = true species, colour = predicted cluster
species_style = [("Adelie", "o", 50), ("Gentoo", "s", 30), ("Chinstrap", "*", 70)]

plt.figure(figsize=(6, 4.5))
for species, marker, size in species_style:
    subset = penguin_df[penguin_df["species"] == species]
    plt.scatter(subset["bill_length_mm"], subset["flipper_length_mm"],
                c=subset["cluster"], s=size, marker=marker, label=species,
                cmap="cividis", edgecolors="k", linewidths=0.5,
                vmin=0, vmax=2)        # same colour scale on all three calls
plt.xlabel("Bill length (mm)")
plt.ylabel("Flipper length (mm)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(pd.crosstab(penguin_df["species"], penguin_df["cluster"]))

# Gentoo comes out almost perfectly: the squares are nearly all one colour, because long
# flippers separate them from the other two. Adelie and Chinstrap overlap in flipper length
# and are split by bill length alone, so each of those shapes carries more than one colour.

**Q5) Now cluster with k = 2 and draw the same figure.**

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.8-target-penguin-kmeans-2.png" alt="The same scatter with only two cluster colours, one of them spanning two species" width="500">

<em>The same plot with k = 2, the value the silhouette score preferred.</em>


The silhouette score preferred k = 2; the data has three species. In a comment, say what that
disagreement means. It is not a bug in either method.

In [ ]:
kmeans_2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
penguin_df["cluster_2"] = kmeans_2.predict(X)

plt.figure(figsize=(6, 4.5))
for species, marker, size in species_style:
    subset = penguin_df[penguin_df["species"] == species]
    plt.scatter(subset["bill_length_mm"], subset["flipper_length_mm"],
                c=subset["cluster_2"], s=size, marker=marker, label=species,
                cmap="cividis", edgecolors="k", linewidths=0.5,
                vmin=0, vmax=1)
plt.xlabel("Bill length (mm)")
plt.ylabel("Flipper length (mm)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(pd.crosstab(penguin_df["species"], penguin_df["cluster_2"]))

# k = 2 scores better because the silhouette score rewards well-separated clusters, and on
# these two measurements the widest gap in the data is the one between Gentoo and everything
# else -- not the one between Adelie and Chinstrap, which barely exists here. The score is
# measuring separation, not the number of species. Neither method is wrong; they answer
# different questions, and only the species column can settle which answer you wanted.

**ℹ️ What you just did**

You chose the number of clusters twice, by two criteria that disagreed, and then checked both
against a ground truth that clustering never saw.

The disagreement is the lesson. The silhouette score measures how well separated the clusters are,
not how many kinds of thing are in the data, and on these two measurements Gentoo sits far enough
from the other two that a clean two-way split scores better than a correct three-way one. k-means
answered the question it was asked. It is worth knowing that the question is not always the one you
meant.

## Exercise 12: Marathon data analysis

The last exercise of Part I is the one dataset in the book that is not environmental, and the only
one where every row is a person. It is here because it exercises the half of seaborn that a
scatter plot and a box plot do not reach: joint distributions, a grid of pairwise relationships,
overlaid densities, and a violin plot split by a second variable.

The data is the finishing record of a marathon — roughly 37 000 runners, each with an age, a
gender, a half-way *split* time and a *final* time. The question it answers is one you can check
against your own intuition, which is what makes it a good last exercise: do people run the second
half of a marathon faster or slower than the first?

Adapted from Jake VanderPlas,
[Python Data Science Handbook](https://github.com/jakevdp/PythonDataScienceHandbook), whose
[marathon-data](https://github.com/jakevdp/marathon-data) repository is the source of the file.

**ℹ️ Beyond this subchapter**

Two of the seaborn functions below are not demonstrated in 1.8's lecture, and are linked where
they are needed:

- [`sns.PairGrid`](https://seaborn.pydata.org/generated/seaborn.PairGrid.html) — Q5, a grid of every
  pairwise combination of a set of columns
- [`sns.violinplot`](https://seaborn.pydata.org/generated/seaborn.violinplot.html) — Q7, a box plot
  whose width shows the distribution

`sns.jointplot` and `sns.kdeplot` are both from the lecture, as is `pd.read_csv`. The `datetime`
module and `read_csv`'s `converters=` argument are used only in the pre-supplied cell.

In [ ]:
# Pre-supplied: download the data file and cache it locally, and parse the two time
# columns into timedeltas on the way in.
import datetime

In [ ]:
import pooch

In [ ]:
path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/marathon-data.csv",
    known_hash="sha256:0fff8a2bdf0cfb8887080c8132fed59399531fb836815c1935cc70823b940a85",
    fname="marathon-data.csv",
    path=pooch.os_cache("mlees"),
)

In [ ]:
def convert_time(s):
    """Turn an 'hh:mm:ss' string into a timedelta."""
    hours, minutes, seconds = map(int, s.split(":"))
    return datetime.timedelta(hours=hours, minutes=minutes, seconds=seconds)

In [ ]:
data = pd.read_csv(path, converters={"split": convert_time, "final": convert_time})
data.head()

**Q1) Add two columns, `split_sec` and `final_sec`, holding the same two times in seconds.**

A `timedelta` column is awkward to plot and awkward to divide. Convert both with
`.dt.total_seconds()`, then print the first few rows to check.

**⚠️ Why not `.astype(int) / 1e9`?**

Earlier versions of this exercise converted with `data["split"].astype(int) / 1e9`, which reads the
integer underneath the timedelta and divides by a billion — correct only while pandas stored
timedeltas in nanoseconds. Current pandas stores them in microseconds, so that expression now
returns a number a thousand times too small, silently and without an error.

`.dt.total_seconds()` asks for the quantity you want rather than for the bytes that happen to
represent it, and keeps working when the representation changes. The split *fraction* in Q3 is a
ratio, so it survives either way — which is exactly what would have let the bug through.

In [ ]:
data["split_sec"] = data["split"].dt.total_seconds()
data["final_sec"] = data["final"].dt.total_seconds()
print(data.head())
print(data[["split_sec", "final_sec"]].describe().round(0))

# split times run from 3921 s (about 65 minutes) to 17989 s, and finals from 7731 s to
# 36068 s -- the right order of magnitude for a marathon, which is the check worth doing
# before any of this is plotted.

**Q2) Use `sns.jointplot` to plot `final_sec` against `split_sec`.**

Use `kind="hex"` for the two-dimensional histogram, and add the line a runner of perfectly even
pace would lie on:

```python
g.ax_joint.plot(np.linspace(4000, 16000), np.linspace(8000, 32000), ":k")
```

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.8-target-marathon-jointplot.png" alt="A hexagonal-bin joint distribution of final time against split time, the dense band lying above a dotted line of even pace" width="450">

<em>The figure to replicate. The dotted line is where a runner whose second half took exactly as long
as the first would fall.</em>


The distribution lies almost entirely *above* the line: most people slow down over the course of
the race. Runners who do the opposite are said to have *negative-split* the race.

In [ ]:
with sns.axes_style("white"):
    g = sns.jointplot(x="split_sec", y="final_sec", data=data, kind="hex")
    # where a runner of perfectly even pace would lie: final = 2 x split
    g.ax_joint.plot(np.linspace(4000, 16000), np.linspace(8000, 32000), ":k")
plt.show()

# Almost the whole distribution sits above the line, and the gap widens for slower runners:
# the longer the race takes, the more the second half costs relative to the first.

**Q3) Add a `split_frac` column measuring how much each runner sped up or slowed down.**

$$\text{split\_frac} = 1 - 2 \times \frac{\text{split\_sec}}{\text{final\_sec}}$$

It is zero for an exactly even race, negative for a negative split, and positive for slowing down.

In [ ]:
data["split_frac"] = 1 - 2 * data["split_sec"] / data["final_sec"]
print(data.head())

**Q4) Print how many runners had a `split_frac` below zero.**

You should get 251 — out of nearly 37 000.

In [ ]:
n_negative = (data["split_frac"] < 0).sum()
print(n_negative, "of", len(data), "runners negative-split the race")   # 251 of 37250

**Q5) Use `sns.PairGrid` to look for what `split_frac` correlates with.**

Build the grid over `age`, `split_sec`, `final_sec` and `split_frac`, with `hue="gender"` so men and
women are drawn separately, and `palette="RdBu_r"`. Map `plt.scatter` over it and add a legend.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.8-target-marathon-pairgrid.png" alt="A four-by-four grid of pairwise scatter plots of age, split time, final time and split fraction, coloured by gender" width="100%">

<em>The figure to replicate.</em>


In a comment, name the one pair in the grid that is a near-perfect straight line, and say why that
particular relationship is not a finding about runners.

In [ ]:
g = sns.PairGrid(data, vars=["age", "split_sec", "final_sec", "split_frac"],
                 hue="gender", palette="RdBu_r")
g.map(plt.scatter, alpha=0.8, s=4)
g.add_legend()
plt.show()

print(data[["age", "split_sec", "final_sec", "split_frac"]].corr().round(3))

# split_sec against final_sec is the near-straight line, at r = 0.956. That is not a fact
# about runners: the split time is the first half of the final time, so the two share most
# of their content by construction. A tight relationship between a part and the whole it
# belongs to is arithmetic, not a result.

**Q6) Compare the split-fraction distributions of men and women with `sns.kdeplot`.**

One filled density per gender on the same axes, labelled, with a legend.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.8-target-marathon-kde.png" alt="Two overlaid density curves of split fraction, one per gender, both centred above zero" width="450">

<em>The figure to replicate.</em>


Hint: seaborn renamed this argument — it is `fill=True` now, not `shade=True`.

In [ ]:
plt.figure(figsize=(6, 4))
sns.kdeplot(data.loc[data["gender"] == "M", "split_frac"], label="men", fill=True)
sns.kdeplot(data.loc[data["gender"] == "W", "split_frac"], label="women", fill=True)
plt.legend()
plt.xlabel("split fraction")
plt.show()

# Both distributions sit clearly to the right of zero -- almost everyone slows down -- and
# the two overlap heavily, with the men's slightly wider.

**Q7) Split the comparison by age as well, with `sns.violinplot`.**

Add an `age_decade` column with `10 * (data["age"] // 10)`, then draw `split_frac` against
`age_decade` with `hue="gender"`. `split=True` puts the two genders on either side of the same
violin, which is what makes them comparable decade by decade.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.8-target-marathon-violin.png" alt="Split violins of split fraction by age decade, each violin divided between the two genders" width="500">

<em>The figure to replicate.</em>


In a comment: the violins for the oldest decades are much narrower than the rest. Before reading
that as "older runners pace more consistently", check how many runners are in those decades.

In [ ]:
data["age_decade"] = 10 * (data["age"] // 10)

plt.figure(figsize=(8, 4.5))
sns.violinplot(data=data, x="age_decade", y="split_frac", hue="gender",
               split=True, inner="quartile", palette="RdBu_r")
plt.xlabel("age decade")
plt.ylabel("split fraction")
plt.show()

print(data.groupby("age_decade")["split_frac"].count())

# The counts are the point: 12 263 runners in their forties, 151 in their seventies, 15 in
# their eighties. The narrow violins on the right are narrow because there is almost nothing
# in them, not because those runners pace themselves more evenly. A violin plot shows shape
# at any sample size and never shows n, so the count has to be read separately.

**ℹ️ What you just did**

You derived a quantity that was not in the file — the split fraction — from two that were, then
looked at it four ways: against its own inputs, against every other column at once, split by one
categorical variable, then by two.

The Q7 warning is the one to carry forward. A violin plot shows you the shape of a distribution
and tells you nothing about how many points went into it, so a narrow violin over the fifteen runners in
their eighties looks much like a confident result over the twelve thousand in their forties. Counting the rows behind a group is not an
extra check to run when something looks surprising; it is part of reading the plot.

That closes Part I. Everything after this is machine learning proper — but it is built on exactly
what you have been doing here: get the data, check what is actually in it, derive what you need,
and plot it before you believe it.